# Context Management in the Claude API

As conversations grow longer, three compounding challenges emerge:
- **Cost**: larger inputs cost more per API call
- **Latency**: longer context takes more time to process
- **Quality**: accuracy and recall degrade as token count increases — a phenomenon Anthropic calls *context rot*

This notebook covers five core techniques:

| Technique | Problem Solved |
|-----------|----------------|
| **1. Prompt Caching** | Cost — reuse repeated context at 10% of base price |
| **2. Context Rot Mitigation** | Quality — strategic positioning preserves recall accuracy |
| **3. Token Budget Management** | Visibility — track and predict context usage before hitting limits |
| **4. Conversation Compaction** | Continuity — reset context while preserving task history |
| **5. Session Memory Compaction** | UX — zero-wait compaction via proactive background processing |

Each section includes a working TypeScript example you can run top-to-bottom.

## Prerequisites

This notebook uses the **Deno kernel**. Packages are imported via `npm:` — no install step needed.

Create a `.env` file at the project root:
```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') });
const MODEL = 'claude-sonnet-4-6';

// Shared helpers used across sections
type SimpleMsg = { role: 'user' | 'assistant'; content: string };

/** Converts SimpleMsg[] to Anthropic.MessageParam[], adding cache_control to the last user message. */
function toCachedApiMessages(messages: SimpleMsg[]): Anthropic.MessageParam[] {
  let lastUserIdx = -1;
  messages.forEach((m, i) => { if (m.role === 'user') lastUserIdx = i; });

  return messages.map((m, i): Anthropic.MessageParam => {
    if (i === lastUserIdx) {
      return {
        role: 'user',
        content: [{ type: 'text', text: m.content, cache_control: { type: 'ephemeral' } }],
      } as Anthropic.MessageParam;
    }
    return { role: m.role, content: m.content };
  });
}

console.log('Setup complete. Model:', MODEL);

---
## 1. Prompt Caching

Prompt caching lets Claude reuse previously processed context. Once a prefix is cached, subsequent requests that share that prefix pay a fraction of the normal cost.

### How to Enable

Add `cache_control: { type: 'ephemeral' }` to a content block to mark it as a **cache breakpoint**. Claude caches everything up to and including that block.

```typescript
messages: [{
  role: 'user',
  content: [{
    type: 'text',
    text: `<document>\n${largeDoc}\n</document>\n\nAnswer: ${question}`,
    cache_control: { type: 'ephemeral' },  // ← cache breakpoint
  }],
}]
```

### Cost Structure

| Operation | Multiplier | Practical Effect |
|-----------|-----------|------------------|
| Cache write (5-min TTL) | **1.25×** | 25% surcharge on first call |
| Cache write (1-hour TTL) | **2.0×** | Higher cost, use for infrequent prompts |
| **Cache read (hit)** | **0.1×** | **90% cheaper than base price** |
| No cache | 1.0× | Standard input pricing |

The 1-hour TTL uses `cache_control: { type: 'ephemeral', ttl: '1h' }` and requires the `extended-cache-ttl-2025-04-11` beta header.

### Minimum Cacheable Tokens

| Model | Minimum |
|-------|---------|
| Claude Opus 4.7 / 4.6 / 4.5 | **4,096 tokens** |
| Claude Sonnet 4.6 / 4.5 | **1,024 tokens** |
| Claude Haiku 4.5 | **4,096 tokens** |

Prompts below the minimum are **silently not cached** — no error is raised. Check `usage` fields to confirm caching occurred.

### Maximum Breakpoints

Up to **4 cache breakpoints** per request. If you add 4 explicit breakpoints, automatic caching returns a 400 error (no slots remaining).

### Reading Usage Stats

```typescript
// ⚠️ input_tokens is ONLY tokens AFTER the last breakpoint — NOT the full input!
response.usage.input_tokens                  // new tokens (after breakpoint)
response.usage.cache_creation_input_tokens   // tokens written to cache
response.usage.cache_read_input_tokens       // tokens read from cache

// Total input processed = sum of all three:
const totalInput = usage.input_tokens + usage.cache_creation_input_tokens + usage.cache_read_input_tokens;
```

In [ ]:
// ── Prompt Caching Demo ───────────────────────────────────────────────────
// A large reference document is cached on the first call.
// The second call (different question, same document) reads from cache.

const REFERENCE_DOC = `
# REST API Design Reference

## HTTP Methods
GET: Retrieve. Idempotent and safe.
POST: Create or trigger. Not idempotent.
PUT: Full replacement. Idempotent.
PATCH: Partial update. Idempotent.
DELETE: Remove. Idempotent.

## Status Codes
200 OK: Successful retrieval or update.
201 Created: Resource successfully created. Include Location header.
204 No Content: Success with no body (e.g., DELETE).
400 Bad Request: Invalid syntax or parameters.
401 Unauthorized: Authentication required or invalid.
403 Forbidden: Authenticated but not permitted.
404 Not Found: Resource does not exist.
409 Conflict: State conflict (e.g., duplicate unique field).
422 Unprocessable Entity: Validation failure on correct syntax.
429 Too Many Requests: Rate limit exceeded. Include Retry-After.
500 Internal Server Error: Unexpected server failure.
503 Service Unavailable: Temporary downtime. Include Retry-After.

## Authentication
API Keys: Server-to-server. Pass in Authorization: Bearer <key>.
OAuth 2.0: Delegated user access. Use PKCE for public clients.
JWT: Self-contained tokens. Verify signature and expiry on every request.

## Pagination
Cursor-based: Opaque cursor for stable ordering. Best for large, frequently updated datasets.
Offset-based: page + per_page. Simple but fragile under concurrent writes.
Link headers: next/prev/first/last (RFC 5988) for discoverability.

## Rate Limiting
Headers: X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset.
On 429: return Retry-After header with seconds until window resets.
`.repeat(4); // ~1,300 tokens — exceeds Sonnet 4.6's 1,024 minimum

function logUsage(usage: Anthropic.Usage, label: string): void {
  const write  = (usage as Record<string, number>).cache_creation_input_tokens ?? 0;
  const read   = (usage as Record<string, number>).cache_read_input_tokens ?? 0;
  const newTok = usage.input_tokens;
  const total  = newTok + write + read;

  console.log(`\n${label}`);
  console.log(`  Cache write : ${write.toLocaleString()} tokens  (×1.25 cost)`);
  console.log(`  Cache read  : ${read.toLocaleString()} tokens  (×0.10 cost — 90% savings)`);
  console.log(`  New input   : ${newTok.toLocaleString()} tokens  (×1.00 cost)`);
  console.log(`  Total input : ${total.toLocaleString()} tokens`);
  if (read > 0)        console.log(`  ✓ Cache HIT — ${((read / total) * 90).toFixed(0)}% cost saved on ${((read / total) * 100).toFixed(0)}% of input`);
  else if (write > 0)  console.log(`  → Cache WRITTEN — next call with same prefix will be 90% cheaper`);
  else                 console.log(`  ✗ No cache activity — prompt may be below minimum token threshold`);
}

// Call 1: writes to cache
const call1 = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{
    role: 'user',
    content: [{
      type: 'text',
      text: `<reference>\n${REFERENCE_DOC}\n</reference>\n\nList the HTTP methods and their idempotency.`,
      cache_control: { type: 'ephemeral' },
    }],
  }],
});
logUsage(call1.usage, 'Call 1 — first request (cache write):');

// Call 2: same document prefix → cache read
const call2 = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{
    role: 'user',
    content: [{
      type: 'text',
      text: `<reference>\n${REFERENCE_DOC}\n</reference>\n\nWhat status code means "resource not found"?`,
      cache_control: { type: 'ephemeral' },
    }],
  }],
});
logUsage(call2.usage, 'Call 2 — same document, different question (cache read):');
console.log('\nAnswer:', call2.content[0].type === 'text' ? call2.content[0].text : '');

In [ ]:
// ── Multi-Turn Caching Pattern ────────────────────────────────────────────
// Place cache_control on the LAST user message each turn.
// This caches the entire conversation history, so each subsequent call
// reads the growing history from cache instead of reprocessing it.

const conversation: SimpleMsg[] = [];
const userTurns = [
  'Design the main resources for a REST API for an e-commerce platform.',
  'Detail the endpoints for the /orders resource — methods, paths, and payloads.',
  'What authentication strategy is best for mobile clients accessing this API?',
];

let turnNum = 0;
for (const msg of userTurns) {
  turnNum++;
  conversation.push({ role: 'user', content: msg });

  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    messages: toCachedApiMessages(conversation),
  });

  const reply = response.content[0].type === 'text' ? response.content[0].text : '';
  conversation.push({ role: 'assistant', content: reply });

  const read  = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
  const write = (response.usage as Record<string, number>).cache_creation_input_tokens ?? 0;
  const total = response.usage.input_tokens + read + write;
  const hitPct = total > 0 ? ((read / total) * 100).toFixed(0) : '0';

  console.log(`Turn ${turnNum} | total_input=${total.toLocaleString()} | cache_read=${read.toLocaleString()} | hit=${hitPct}% | history_msgs=${conversation.length}`);
}

console.log('\n✓ Cache hit rate grows each turn as conversation history accumulates in cache.');

---
## 2. Context Rot — The "Lost in the Middle" Effect

As context grows, *context rot* sets in: accuracy and recall degrade for information buried in the **middle** of a long context window. More context is not automatically better — what's *in* context and *where* it's placed matters significantly.

### The Effect

Long-context benchmarks (MRCR, GraphWalks) show a characteristic recall pattern:

```
Recall accuracy
  ↑
  █                                                         █
  █   ▇                                               ▇   █
  █   ▇   ▅                                     ▅   ▇   █
  █   ▇   ▅   ▃   ▂   ▂   ▂   ▂   ▂   ▃   ▅   ▇   █
  └─────────────────────────────────────────────────────────→ Position
  Start                   Middle                           End
```

The degradation is minor at < 10K tokens but becomes significant at 50K–200K+ tokens.

### Anthropic's Official Guidance

> *"Place your long documents and inputs (~20K+ tokens) near the **top** of your prompt, above your query, instructions, and examples. Queries at the **end** can improve response quality by up to 30% in tests, especially with complex, multi-document inputs."*

### Mitigation Strategies

| Strategy | How to Apply | When to Use |
|----------|-------------|-------------|
| **Strategic positioning** | Documents at top; query + constraints at end | Always, for any long prompt |
| **Document tagging** | Wrap each doc in `<document>` with `<source>` metadata | Multiple documents |
| **Quote-first** | Ask Claude to quote the relevant passage before answering | Noisy or long documents |
| **Retrieval (RAG)** | Inject only relevant chunks, not the full corpus | Large document sets |
| **Compaction** | Summarise old turns; keep only recent turns intact | Long multi-turn conversations |

In [ ]:
// ── Context Rot Mitigation: Document Positioning ──────────────────────────
// Correct pattern: documents at the TOP, query at the BOTTOM.

const DOCS = [
  {
    source: 'rate-limiting-spec.md',
    content: 'Token bucket algorithm. Capacity: 500 tokens. Refill: 100/second. Burst: 20% above limit for 10s. Header: X-RateLimit-Remaining.',
  },
  {
    source: 'security-policy.md',
    content: 'API keys rotate every 90 days. Keys unused 30+ days are auto-revoked. Compromise must be reported within 24 hours.',
  },
  {
    source: 'deployment-runbook.md',
    content: 'Pre-deploy: run tests → update CHANGELOG → tag git release → notify PagerDuty → deploy staging → smoke test → deploy prod.',
  },
];

// ❌ Anti-pattern: query first, documents below
const badPromptStructure = [
  'What is the rate limiting strategy?',
  '',
  ...DOCS.map(d => `[${d.source}]\n${d.content}`),
].join('\n');

// ✅ Correct pattern: documents first (tagged), query last
function buildLongContextPrompt(docs: typeof DOCS, query: string): string {
  const taggedDocs = docs
    .map(d => `<document>\n<source>${d.source}</source>\n<document_content>\n${d.content}\n</document_content>\n</document>`)
    .join('\n\n');
  return `${taggedDocs}\n\n${query}`;
}

const query = 'Based on the documents above, summarise the rate limiting approach.';
const goodPrompt = buildLongContextPrompt(DOCS, query);

const response = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{ role: 'user', content: goodPrompt }],
});

console.log('Prompt structure used:');
console.log('  [<document> doc1 </document>] [<document> doc2 </document>] ... [QUERY]');
console.log('  ← documents at top ──────────────────────── query at end →\n');
console.log('Response:', response.content[0].type === 'text' ? response.content[0].text : '');
console.log('\n✓ Documents wrapped in <document> tags for clear structure');
console.log('✓ Query placed last — closest to where Claude generates its response');
console.log('\nAnti-pattern to avoid: placing query at the top, documents below.');

In [ ]:
// ── Quote-First Pattern ────────────────────────────────────────────────────
// Instruct Claude to quote the relevant passage BEFORE answering.
// This forces active retrieval from context and reduces hallucination
// on large documents where context rot is most likely.

const ARCH_DOC = `
# System Architecture

## Authentication Service
RS256-signed JWTs. Access tokens expire after 15 minutes. Refresh tokens use HS256 and expire after 7 days.
Invalid refresh tokens cause immediate session termination. Tokens are validated on every request.

## Data Pipeline
Kafka with 8 partitions. Throughput: 50,000 events/second. Consumer groups maintain independent offsets.
Lag alert threshold: 10,000 unprocessed events.

## Cache Layer
Redis cluster: 3 primary, 3 replica nodes. Write-through on all user-facing reads.
TTL: 5 min (session data), 1 hour (product catalog), 24 hours (user preferences).

## Rate Limiting
Token bucket. Capacity: 500 tokens, refill: 100/second.
Endpoints under /api/v1/ subject to per-IP limits. Endpoints under /api/v2/ exempt from per-IP but subject to per-account limits.
`.repeat(5);

// Quote-first instruction embedded at the end of the prompt
const quoteFirstPrompt = buildLongContextPrompt(
  [{ source: 'architecture.md', content: ARCH_DOC }],
  'First, quote the exact sentences about JWT expiry.\nThen answer: how long do access tokens last, and what happens when a refresh token is invalid?'
);

const quoteResp = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  messages: [{ role: 'user', content: quoteFirstPrompt }],
});

console.log('Quote-first response:');
console.log(quoteResp.content[0].type === 'text' ? quoteResp.content[0].text : '');
console.log('\n✓ Claude quotes the relevant passage before interpreting it');
console.log('  → Forces active retrieval, prevents context-rot-induced confabulation');

---
## 3. Token Budget Management

### Context Window Sizes

| Model | Context Window | Max Output |
|-------|---------------|------------|
| Claude Opus 4.7 / 4.6 | **1,000,000 tokens** | 32,000 tokens |
| Claude Sonnet 4.6 | **1,000,000 tokens** | 64,000 tokens |
| Claude Sonnet 4.5 / Haiku 4.5 | 200,000 tokens | 8,000–16,000 tokens |

> Claude 4 models (Opus 4.7/4.6, Sonnet 4.6) have a **1M token** context window.

### Token Counting API

Before sending a large request, estimate token usage to avoid hitting limits unexpectedly:

```typescript
const count = await client.messages.countTokens({
  model: 'claude-sonnet-4-6',
  messages: myMessages,
});
console.log(count.input_tokens); // estimated tokens for this request
```

### Context Overflow Behaviour

| Model Generation | Behaviour when `input_tokens + max_tokens > context_window` |
|-----------------|-------------------------------------------------------------|
| Claude 4.5+ | API **accepts** the request; stops generation with `stop_reason: "model_context_window_exceeded"` |
| Earlier models | API returns a **validation error** before processing |

### Context Awareness (Sonnet 4.6, Sonnet 4.5, Haiku 4.5)

These models automatically receive their remaining token budget, injected by the API — no developer action needed:

```xml
<!-- Injected at conversation start -->
<budget:token_budget>1000000</budget:token_budget>

<!-- Updated after each tool call -->
<system_warning>Token usage: 35000/1000000; 965000 remaining</system_warning>
```

This lets models self-regulate verbosity and tool usage as context fills, making them more reliable in long-running agentic tasks.

In [ ]:
// ── Token Budget Management ───────────────────────────────────────────────

const CONTEXT_WINDOW = 1_000_000; // Sonnet 4.6 (1M token context window)
const COMPACT_AT     = 0.75;      // trigger compaction at 75% full

interface Budget {
  totalInput : number;
  output     : number;
  pctUsed    : number;
  remaining  : number;
  mustCompact: boolean;
}

function getBudget(usage: Anthropic.Usage): Budget {
  const read   = (usage as Record<string, number>).cache_read_input_tokens ?? 0;
  const write  = (usage as Record<string, number>).cache_creation_input_tokens ?? 0;
  const input  = usage.input_tokens + read + write;
  const total  = input + usage.output_tokens;
  return {
    totalInput : input,
    output     : usage.output_tokens,
    pctUsed    : (total / CONTEXT_WINDOW) * 100,
    remaining  : CONTEXT_WINDOW - total,
    mustCompact: total / CONTEXT_WINDOW >= COMPACT_AT,
  };
}

function printBudget(b: Budget, label: string): void {
  const filled = Math.round(b.pctUsed / 5);
  const bar    = '█'.repeat(filled) + '░'.repeat(20 - filled);
  console.log(`\n${label}`);
  console.log(`  [${bar}] ${b.pctUsed.toFixed(3)}%`);
  console.log(`  Input: ${b.totalInput.toLocaleString()} | Output: ${b.output.toLocaleString()} | Remaining: ${b.remaining.toLocaleString()}`);
  if (b.mustCompact) console.log('  ⚠️  Approaching limit — initiate compaction!');
  else               console.log('  ✅  Within budget');
}

// Step 1: Pre-flight estimate using the token counting API
const preflightMessages: Anthropic.MessageParam[] = [{
  role: 'user',
  content: 'Explain the trade-offs between REST and GraphQL APIs in 3 paragraphs.',
}];

const estimate = await client.messages.countTokens({ model: MODEL, messages: preflightMessages });
console.log(`Pre-flight estimate: ${estimate.input_tokens} input tokens`);
console.log(`Context capacity before this call: ${(CONTEXT_WINDOW - estimate.input_tokens).toLocaleString()} tokens remaining`);

// Step 2: Send the request and measure actual usage
const response = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  messages: preflightMessages,
});

const budget = getBudget(response.usage);
printBudget(budget, 'After turn 1:');

// Step 3: Check for context overflow signal
if (response.stop_reason === 'model_context_window_exceeded') {
  console.log('\n⛔ Context window exceeded — response was truncated.');
  console.log('   Initiate compaction and retry the last turn.');
} else {
  console.log(`\nstop_reason: "${response.stop_reason}" — generation completed normally.`);
}

// Dynamic max_tokens: cap at 50% of remaining capacity
const safeMaxTokens = Math.min(8_192, Math.floor(budget.remaining * 0.5));
console.log(`\nRecommended max_tokens for next call: ${safeMaxTokens.toLocaleString()}`);

---
## 4. Conversation Compaction

When a conversation approaches the context limit, compaction replaces the message history with a concise summary, allowing the conversation to continue indefinitely.

### Two Approaches

**Manual Compaction** — you implement the logic:
1. Track token usage after each API call
2. When usage exceeds a threshold, call Claude to generate a structured summary
3. Replace message history with the summary
4. Continue from the summary

**SDK Automatic Compaction** — via `client.beta.messages.toolRunner()`:
- Available in `@anthropic-ai/sdk >= 0.50`
- Supported models: Claude Mythos Preview, Opus 4.7, Opus 4.6, Sonnet 4.6
- SDK monitors token usage per turn and handles summarisation automatically

```typescript
const runner = client.beta.messages.toolRunner(MODEL, {
  maxTokens: 4096,
  tools: myTools,
  messages: initialMessages,
  compactionControl: {
    enabled: true,
    contextTokenThreshold: 50_000,       // trigger at 50K tokens
    model: 'claude-haiku-4-5',           // cheaper model for summaries
    summaryPrompt: 'Custom instructions...',  // optional
  },
});

for await (const message of runner) {
  // Each iteration is one API turn
  // Compaction happens automatically when threshold is exceeded
  console.log('Turn complete. Compactions so far:', runner.compactionCount);
}
```

> **Note**: `compactionControl` works ONLY with `client.beta.messages.toolRunner()` — not with `client.messages.create()`. It is designed for tool-use agentic workflows, not plain chat.

### Effective Compaction Prompt

A well-structured compaction prompt should preserve:
- The user's original goal and any stated constraints
- Completed work with specific details (names, IDs, paths, values)
- Errors encountered and how they were resolved
- What was actively in progress at the moment of compaction
- Pending tasks not yet started

In [ ]:
// ── Manual Conversation Compaction ────────────────────────────────────────

const COMPACT_THRESHOLD_TOKENS = 4_000; // low value for demo

const COMPACTION_PROMPT = `Create a structured continuation memory for this conversation.

## Goal
The user's original request and any constraints they stated.

## Completed
Specific outputs produced (names, IDs, values, decisions made).

## In Progress
What was being actively worked on at the end of the transcript.

## Pending
Items requested but not yet started.

## Key References
Important identifiers, paths, or values needed to continue.

Write concisely in past tense. Omit pleasantries. Output the summary only.`;

class ManualCompactionSession {
  private messages: SimpleMsg[] = [];
  private inputTokens = 0;
  private compactionCount = 0;

  constructor(private readonly system = '') {}

  async send(userMessage: string): Promise<string> {
    this.messages.push({ role: 'user', content: userMessage });

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      ...(this.system ? { system: this.system } : {}),
      messages: toCachedApiMessages(this.messages),
    });

    const reply = response.content[0].type === 'text' ? response.content[0].text : '';
    this.messages.push({ role: 'assistant', content: reply });

    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    this.inputTokens = response.usage.input_tokens + cacheRead;

    const statusIcon = this.inputTokens >= COMPACT_THRESHOLD_TOKENS ? '⚠️ ' : '✅';
    console.log(`  [msgs=${this.messages.length} | input=${this.inputTokens.toLocaleString()} tokens | ${statusIcon}]`);

    if (this.inputTokens >= COMPACT_THRESHOLD_TOKENS) {
      await this.compact();
    }
    return reply;
  }

  private async compact(): Promise<void> {
    this.compactionCount++;
    const summaryResp = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      messages: [...toCachedApiMessages(this.messages), { role: 'user', content: COMPACTION_PROMPT }],
    });
    const summary = summaryResp.content[0].type === 'text' ? summaryResp.content[0].text : '';
    this.messages = [{
      role: 'user',
      content: `[Compaction #${this.compactionCount} — continuation memory]:\n${summary}\n\nContinue from where we left off.`,
    }];
    console.log(`\n  🔄 Compacted to 1 message (was ${this.messages.length + 2} messages)`);
  }
}

const compactSession = new ManualCompactionSession('You are a concise API design consultant.');

const designTurns = [
  'Design a REST API for a task management app. What resources are needed?',
  'Detail the /tasks endpoint — operations, fields, and error responses.',
  'Should tasks support nested subtasks? How would that work in REST?',
  'What about real-time updates — WebSockets or Server-Sent Events?',
  'Finalize the authentication: JWT or session tokens? Why?',
];

for (const turn of designTurns) {
  console.log(`\nUser: ${turn}`);
  const reply = await compactSession.send(turn);
  console.log(`Claude: ${reply.substring(0, 100)}...`);
}

---
## 5. Session Memory Compaction

Standard compaction pauses the user while generating a summary. **Session memory compaction** eliminates this wait by building the summary *proactively in the background* as the conversation continues.

### Two Patterns

**Traditional** (reactive — user must wait):
```
Turn 1 → Turn 2 → Turn N → LIMIT HIT!
                               ↓
                  [user waits ~30s for summary]
                               ↓
                           Continue
```

**Instant** (proactive — zero wait time):
```
Turn 1 → Turn 2 → [soft threshold] → Turn 3 → ... → Turn N → LIMIT!
                        ↓                                        ↓
              [fire background Promise]          [memory already ready]
              [no await — user unaffected]                       ↓
                        ↓                              ⚡ INSTANT SWAP
              📝 Summary builds silently
```

### TypeScript vs Python

| Concept | Python | TypeScript |
|---------|--------|------------|
| Background execution | `threading.Thread(daemon=True).start()` | Fire-and-forget `Promise` (no `await`) |
| Thread safety | `threading.Lock()` | Single-threaded event loop — **no locks needed** |
| Shared state update | `with self._lock: self.memory = x` | Direct assignment after `await` completes |

### Prompt Caching Integration

The background summariser reads the same conversation prefix as the main chat. Adding `cache_control` to the last user message in both calls yields an automatic cache hit on the background call — reducing its cost by ~90%.

```
Main chat       → [system][messages...][last user + cache_control] → Claude
Background call → [system][messages...][last user + cache_control][SUMMARISE prompt]
                            ↑ CACHE HIT — ~90% cheaper for the background call
```

In [ ]:
// ── Session Memory Prompt ─────────────────────────────────────────────────
// Shared by both Traditional and Instant patterns.

const SESSION_MEMORY_PROMPT = `
Compress this conversation into structured memory for seamless task continuation.
Optimise for an AI assistant resuming the task — not for human readability.

## Goal
The user's original request and any clarifications or constraints. Use direct quotes for key requirements.

## Completed Work
Actions successfully performed. Be specific:
- What was created, modified, or analysed
- Exact identifiers (file paths, IDs, URLs, names, values)

## Errors & Corrections
Problems encountered and how they were resolved. Approaches that failed (so they are not retried).
User corrections verbatim — these represent learned preferences.

## Active Work
What was in progress when the session ended, including partial results.

## Pending Tasks
Items the user requested that have not yet been started.

## Key References
Important identifiers, paths, values, or constraints to carry forward.

Rules: weight recent messages heavily; omit pleasantries; keep each section under 300 words.`;

console.log('SESSION_MEMORY_PROMPT defined — used by both Traditional and Instant patterns.');

In [ ]:
// ── Traditional Session Memory Compaction ─────────────────────────────────
// Compacts on-demand when the hard token limit is reached.
// The user must wait for the summary before their next message is processed.

class TraditionalSession {
  private messages: SimpleMsg[] = [];
  private contextTokens = 0;
  private compactions = 0;

  constructor(
    private readonly system: string,
    private readonly contextLimit = 8_000,
  ) {}

  async chat(userMessage: string): Promise<string> {
    if (this.contextTokens >= this.contextLimit) {
      const start = performance.now();
      console.log('\n  ⏳ Context limit hit — compacting (user must wait)...');
      await this.compact();
      console.log(`  Compaction took ${((performance.now() - start) / 1_000).toFixed(1)}s`);
    }

    this.messages.push({ role: 'user', content: userMessage });

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      system: this.system,
      messages: toCachedApiMessages(this.messages),
    });

    const reply = response.content[0].type === 'text' ? response.content[0].text : '';
    this.messages.push({ role: 'assistant', content: reply });

    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    this.contextTokens = response.usage.input_tokens + cacheRead + response.usage.output_tokens;
    return reply;
  }

  private async compact(): Promise<void> {
    this.compactions++;
    const summaryResp = await client.messages.create({
      model: MODEL,
      max_tokens: 2048,
      system: this.system,
      messages: [...toCachedApiMessages(this.messages), { role: 'user', content: SESSION_MEMORY_PROMPT }],
    });
    const summary = summaryResp.content[0].type === 'text' ? summaryResp.content[0].text : '';
    this.messages = [{
      role: 'user',
      content: `[Session memory — compaction #${this.compactions}]:\n${summary}\n\nContinue where we left off.`,
    }];
    this.contextTokens = summaryResp.usage.output_tokens;
    console.log(`  📝 Memory written (compaction #${this.compactions})`);
  }
}

console.log('TraditionalSession defined. Run the next cell to compare with InstantSession.');

In [ ]:
// ── Instant Session Memory Compaction ─────────────────────────────────────
// Builds session memory in the background (fire-and-forget Promise).
// When the hard limit is hit, the summary is already ready — zero wait time.

class InstantSession {
  private messages: SimpleMsg[] = [];
  private contextTokens = 0;
  private sessionMemory: string | null = null;
  private lastSummarisedIdx = 0;

  // null = idle; non-null = background update running
  private updatePromise: Promise<void> | null = null;

  constructor(
    private readonly system: string,
    private readonly hardLimit = 12_000,
    private readonly softLimit  =  6_000,
  ) {}

  async chat(userMessage: string): Promise<string> {
    // Hard limit: swap in the pre-built session memory
    if (this.contextTokens >= this.hardLimit) {
      if (this.updatePromise !== null) {
        console.log('  ⏳ Background memory still building — waiting briefly...');
        await this.updatePromise; // only blocks if background isn't done
      }
      this.swapMemory();
    }

    this.messages.push({ role: 'user', content: userMessage });

    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      system: this.system,
      messages: toCachedApiMessages(this.messages),
    });

    const reply = response.content[0].type === 'text' ? response.content[0].text : '';
    this.messages.push({ role: 'assistant', content: reply });

    const cacheRead = (response.usage as Record<string, number>).cache_read_input_tokens ?? 0;
    this.contextTokens = response.usage.input_tokens + cacheRead + response.usage.output_tokens;

    // Soft limit: trigger background memory build — NO await (fire and forget)
    if (this.contextTokens >= this.softLimit && this.updatePromise === null) {
      console.log('  [Background] Proactively building session memory...');
      const snapshot = [...this.messages];
      // ← Key: no await. This Promise runs concurrently on the event loop.
      this.updatePromise = this.buildMemory(snapshot)
        .then(mem => {
          this.sessionMemory = mem;
          this.lastSummarisedIdx = snapshot.length;
          console.log('  [Background] ✓ Session memory ready — next compaction will be instant.');
        })
        .catch(err => console.error('[Background error]', err))
        .finally(() => { this.updatePromise = null; });
    }

    return reply;
  }

  private async buildMemory(messages: SimpleMsg[]): Promise<string> {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 2048,
      system: this.system,
      // Adding SESSION_MEMORY_PROMPT as a new user turn AFTER the cached prefix.
      // The background call shares the same prefix as the main chat → cache hit → ~90% cheaper.
      messages: [...toCachedApiMessages(messages), { role: 'user', content: SESSION_MEMORY_PROMPT }],
    });
    return response.content[0].type === 'text' ? response.content[0].text : '';
  }

  private swapMemory(): void {
    if (!this.sessionMemory) {
      console.log('  ⚠️  No pre-built memory — this compaction will block.');
      return;
    }
    // Keep any messages that arrived AFTER the last summarised point
    const unsummarised = this.messages.slice(this.lastSummarisedIdx);
    this.messages = [
      { role: 'user', content: `[Session memory]:\n${this.sessionMemory}\n\nContinue where we left off.` },
      ...unsummarised,
    ];
    this.lastSummarisedIdx = 1;
    this.contextTokens = 0;
    console.log('\n⚡ INSTANT SWAP — session memory was pre-built, zero wait time!');
  }

  get memoryReady(): boolean { return this.sessionMemory !== null; }
}

// ── Side-by-side demo ────────────────────────────────────────────────────
const WRITER_SYSTEM = 'You are a concise technical writing assistant.';
const DEMO_TURNS = [
  'Help me write API docs for a payment processing service.',
  'Document the POST /payments endpoint — request body, response, and error codes.',
  'Now document POST /payments/{id}/refund with all possible error codes.',
  'Add a webhooks section explaining the payment.completed and payment.failed events.',
];

console.log('=== Traditional Session ===');
const trad = new TraditionalSession(WRITER_SYSTEM, 4_000);
for (const turn of DEMO_TURNS) {
  const reply = await trad.chat(turn);
  console.log(`User: ${turn.substring(0, 55)}...`);
  console.log(`Claude: ${reply.substring(0, 70)}...\n`);
}

console.log('\n=== Instant Session ===');
const inst = new InstantSession(WRITER_SYSTEM, 4_000, 2_000);
for (const turn of DEMO_TURNS) {
  const reply = await inst.chat(turn);
  console.log(`User: ${turn.substring(0, 55)}...`);
  console.log(`Claude: ${reply.substring(0, 70)}...`);
  console.log(`Memory ready: ${inst.memoryReady}\n`);
}

---
## Summary

### Decision Tree

```
Is the same large context sent on every API call?
  └── Yes ──► Prompt Caching  (cache_control: { type: 'ephemeral' })

Is recall quality degrading on long prompts?
  └── Yes ──► Context Rot Mitigation  (documents at top, query at end)

Is the conversation approaching the context window limit?
  ├── Tool-use agentic workflow?
  │     └── Yes ──► SDK compactionControl  (client.beta.messages.toolRunner)
  └── Conversational (no tools)?
        ├── Users can tolerate a pause?
        │     └── Yes ──► Traditional Compaction  (on-demand)
        └── Zero-wait required?
              └── Yes ──► Instant Compaction  (background Promise)
```

### Quick Reference

| Technique | Key API / Parameter | Effect |
|-----------|---------------------|--------|
| Prompt Caching (5-min) | `cache_control: { type: 'ephemeral' }` | 90% cost reduction on cache hits |
| Prompt Caching (1-hour) | `cache_control: { type: 'ephemeral', ttl: '1h' }` | 1-hour TTL; 2× write cost |
| Token Counting | `client.messages.countTokens()` | Pre-flight input token estimate |
| Overflow Signal | `stop_reason: 'model_context_window_exceeded'` | Graceful overflow on Claude 4.5+ |
| SDK Compaction | `compactionControl: { enabled: true, ... }` | Automatic agentic loop compaction |
| Background Memory | Fire-and-forget `Promise` (no `await`) | Zero-wait session memory swap |
| Document Positioning | Documents → top; query + constraints → end | Up to 30% quality improvement |
| Quote-First | Instruction to quote relevant passage before answering | Reduces context-rot hallucination |

### Combining Techniques

These patterns compose well:

- **Prompt Caching + Instant Compaction**: The background summariser shares the same conversation prefix as the main chat. Adding `cache_control` to the last user message in both calls yields a cache hit on the background summariser — ~90% cheaper background updates.

- **Token Counting + Any Compaction**: Use `client.messages.countTokens()` before each send to decide whether to compact proactively, avoiding wasted API calls.

- **Context Rot Mitigation + Any Approach**: Document positioning should always be applied regardless of which compaction strategy you use — they operate at different levels and do not conflict.